In [1]:
# from newsapi import NewsApiClient

# from utils import append_jsonl

# # Init
# newsapi = NewsApiClient(api_key='e2e77f229f214d0c94ac9fb6df146b4d')

# # /v2/top-headlines
# top_headlines = newsapi.get_top_headlines(q='Apple',
#                                           sources='bbc-news,the-verge',
#                                           language='en')

# # /v2/everything
# all_articles = newsapi.get_everything(q='Apple',
#                                       sources='bbc-news,the-verge',
#                                       domains='bbc.co.uk,techcrunch.com',
#                                       from_param='2025-10-13',
#                                       to='2025-10-14',
#                                       language='en',
#                                       sort_by='relevancy',
#                                       page=2)

# # /v2/top-headlines/sources
# sources = newsapi.get_sources()

# append_jsonl(top_headlines, 'sentiment/cache/test/top_headlines.json')
# append_jsonl(all_articles, 'sentiment/cache/test/all_articles.json')
# append_jsonl(sources, 'sentiment/cache/test/sources.json')

import feedparser
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import nltk
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests
from datetime import datetime, timedelta
from urllib.parse import quote_plus
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import warnings

# Download VADER lexicon
nltk.download('vader_lexicon')

# Data Cleaning
def get_clean_financial_data(ticker, start_date, end_date):
    """Download and clean financial data from Yahoo Finance"""
    data = yf.download(ticker, start=start_date, end=end_date)
    if data is None or data.empty:
        return data
    data.columns = data.columns.get_level_values(0)
    data = data.ffill()
    # Check if index has timezone info before trying to localize
    try:
        if hasattr(data.index, 'tz') and getattr(data.index, 'tz', None) is not None:
            data.index = data.index.tz_localize(None)  # type: ignore
    except (AttributeError, TypeError):
        # If tz_localize is not available, skip timezone handling
        pass
    return data

# Stock List
def get_popular_stocks():
    return {
        "Technology": {
            "Apple": "AAPL",
            "Microsoft": "MSFT",
            "Amazon": "AMZN",
            "Alphabet (Google)": "GOOGL",
            "Nvidia": "NVDA",
            "Tesla": "TSLA",
            "Meta (Facebook)": "META"
        },
        "Finance": {
            "JPMorgan Chase": "JPM",
            "Bank of America": "BAC",
            "Visa": "V",
            "Mastercard": "MA",
            "PayPal": "PYPL"
        },
        "Retail": {
            "Walmart": "WMT",
            "Home Depot": "HD",
            "Target": "TGT",
            "Costco": "COST",
            "Lowe's": "LOW"
        },
        "Healthcare": {
            "Johnson & Johnson": "JNJ",
            "Pfizer": "PFE",
            "Moderna": "MRNA",
            "UnitedHealth Group": "UNH",
            "Merck": "MRK"
        }
    }

# Fetch News
def get_stock_news(company_name, ticker, days=90):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    query = quote_plus(f'{company_name} OR {ticker} after:{start_date.strftime("%Y-%m-%d")} before:{end_date.strftime("%Y-%m-%d")}')
    rss_url = f'https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en'
    feed = feedparser.parse(rss_url)
    news_items = []
    for entry in feed.entries:
        try:
            if hasattr(entry, 'published_parsed'):
                parsed_date = datetime(*entry.published_parsed[:6])
                date_str = parsed_date.strftime('%Y-%m-%d %H:%M:%S')
            else:
                date_str = 'N/A'
            news_items.append({
                'date': date_str,
                'title': entry.title,
                'url': entry.link,
                'source': entry.get('source', {}).get('title', 'N/A'),
                'company': company_name,
                'ticker': ticker
            })
        except Exception as e:
            print(f"Error processing entry: {e}")
            continue
    df = pd.DataFrame(news_items)
    if not df.empty and 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['date'])
        df = df.sort_values('date', ascending=False)
    return df

# Plot + Statistical Test
def plot_sentiment_and_price(df, ticker):
    if df.empty:
        print(f"No news to analyze sentiment for {ticker}")
        return

    vader = SentimentIntensityAnalyzer()
    df['sentiment'] = df['title'].apply(lambda text: vader.polarity_scores(text)['compound'])
    df['date_only'] = df['date'].dt.date
    daily_sentiment = df.groupby('date_only')['sentiment'].mean()

    # Fetch cleaned price data
    start = df['date'].min().date()
    end = df['date'].max().date() + timedelta(days=1)
    price_data = get_clean_financial_data(ticker, start, end)
    if price_data is None or price_data.empty:
        print(f"No price data found for {ticker}")
        return

    daily_price = price_data['Close'].copy()
    # Handle different index types safely
    try:
        if hasattr(daily_price.index, 'date'):
            daily_price.index = daily_price.index.date  # type: ignore
        else:
            # For MultiIndex or other index types, convert to date
            daily_price.index = pd.to_datetime(daily_price.index).date  # type: ignore
    except (AttributeError, TypeError):
        # Fallback: convert index to datetime first, then to date
        daily_price.index = pd.to_datetime(daily_price.index).date  # type: ignore

    merged = pd.DataFrame({
        'Sentiment': daily_sentiment,
        'Price': daily_price
    }).dropna()

    if merged.empty:
        print("No overlapping dates between sentiment and price data.")
        return

    # Plot sentiment
    plt.figure(figsize=(12, 5))
    merged['Sentiment'].ewm(span=15).mean().plot(label='Smoothed Sentiment', color='blue', linewidth=2)
    plt.ylabel("Sentiment Score")
    plt.title(f"Sentiment Trend for {ticker}")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot price
    plt.figure(figsize=(12, 5))
    merged['Price'].plot(label='Stock Price', color='green', linewidth=2)
    plt.ylabel("Price")
    plt.title(f"Stock Price Trend for {ticker}")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Statistical Analysis
    merged['Return'] = merged['Price'].pct_change().fillna(0)

    # Create lagged sentiment variables
    for lag in [1, 2, 3]:
        merged[f'Sentiment_Lag{lag}'] = merged['Sentiment'].shift(lag)

    # Drop rows with NaN values from lagging
    merged.dropna(inplace=True)

    # Run OLS for each lag
    print("\n📈 OLS Regression Results by Lag:")
    for lag in [1, 2, 3]:
        X = sm.add_constant(merged[f'Sentiment_Lag{lag}'])
        y = merged['Return']

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=FutureWarning, message="verbose is deprecated since functions should not print results")
            warnings.filterwarnings("ignore", message="kurtosistest only valid for n>=20")
            model = sm.OLS(y, X).fit()
            print(f"\nLag {lag} Results:")
            print(f"Coefficient: {model.params.iloc[1]:.4f}")
            print(f"p-value: {model.pvalues.iloc[1]:.4f}")
            if model.pvalues.iloc[1] < 0.05:
                direction = "positive" if model.params.iloc[1] > 0 else "negative"
                print(f"✅ Significant {direction} impact (p < 0.05)")
            else:
                print("❌ No significant impact (p ≥ 0.05)")

    # Granger Causality Test
    print("\n📊 Granger Causality Test Results:")
    try:
        gc_results = grangercausalitytests(merged[['Return', 'Sentiment']], maxlag=3, verbose=False)
        for lag in gc_results:
            test_stats = gc_results[lag][0]
            f_pval = test_stats['ssr_ftest'][1]
            chi2_pval = test_stats['ssr_chi2test'][1]
            print(f"\nLag {lag}:")
            print(f"F-test p-value: {f_pval:.4f}")
            print(f"Chi2-test p-value: {chi2_pval:.4f}")
            if f_pval < 0.05 or chi2_pval < 0.05:
                print("✅ Significant Granger causality (p < 0.05)")
            else:
                print("❌ No significant Granger causality (p ≥ 0.05)")
    except Exception as e:
        print(f"Granger causality test failed: {e}")

# UI
def create_stock_news_ui():
    stocks = get_popular_stocks()
    result_df = pd.DataFrame()
    initial_category = "Technology"
    initial_stocks = [(f"{k} ({v})", (k, v)) for k, v in stocks[initial_category].items()]

    category_dropdown = widgets.Dropdown(
        options=list(stocks.keys()),
        value=initial_category,
        description='Category:'
    )

    stock_dropdown = widgets.Dropdown(
        options=initial_stocks,
        description='Stock:',
        layout={'width': '500px'}
    )

    days_slider = widgets.IntSlider(
        value=30,
        min=1,
        max=365,
        step=1,
        description='Days:',
        continuous_update=False
    )

    output = widgets.Output()
    fetch_button = widgets.Button(description="Fetch News", button_style='info')

    def update_stock_dropdown(change):
        new_category = change['new']
        stock_dropdown.options = [(f"{k} ({v})", (k, v)) for k, v in stocks[new_category].items()]

    category_dropdown.observe(update_stock_dropdown, names='value')

    def on_fetch_click(b):
        nonlocal result_df
        with output:
            clear_output()
            if stock_dropdown.value:
                company_name, ticker = stock_dropdown.value
                result_df = get_stock_news(company_name, ticker, days_slider.value)
                if not result_df.empty:
                    display_df = result_df.head(5).copy()  # Display only the top 5 news items
                    display_df['date'] = display_df['date'].dt.strftime('%Y-%m-%d %H:%M')
                    display_df['title'] = display_df.apply(
                        lambda x: f'{x["title"]}', axis=1
                    )
                    display(HTML(display_df[['date', 'title', 'source']].to_html(escape=False)))
                    plot_sentiment_and_price(result_df, ticker)
                else:
                    print("No news articles found.")

    fetch_button.on_click(on_fetch_click)

    display(widgets.VBox([
        widgets.HBox([category_dropdown, stock_dropdown]),
        days_slider,
        fetch_button,
        output
    ]))

# Run
print("📈 Stock News Analyzer: Select a company and click 'Fetch News'")
create_stock_news_ui()

📈 Stock News Analyzer: Select a company and click 'Fetch News'


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
